<a href="https://colab.research.google.com/github/Rfeef-Abdulaziz/task4/blob/main/task4_rafeef.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# إعدادات الصفحة الأساسية
st.set_page_config(page_title="تحليل بيانات النوبات القلبية", layout="wide")

@st.cache_data
def load_data():
    df = pd.read_csv("data88.csv")
    return df

try:
    df = load_data()

    # القائمة الجانبيه
    st.sidebar.title(" القائمة الرئيسية")
    page = st.sidebar.radio("اختر الصفحة:", ["نظرة عامة على البيانات", "الرسوم البيانية التفاعلية", " نموذج التنبؤ"])

    # نظرة عامة على البيانات

    if page == "نظرة عامة على البيانات":
        st.title(" لوحة تحكم استكشاف بيانات الأزمات القلبية")
        st.write("تركز هذه الصفحة على إعطاء نظرة سريعة وإحصائية شاملة لملف البيانات الخاص بك.")

        # Metrics
        col1, col2, col3, col4 = st.columns(4)
        with col1:
            st.metric("إجمالي عدد المرضى", f"{df.shape[0]:,}")
        with col2:
            st.metric("عدد المؤشرات الطبية", df.shape[1])
        with col3:
            st.metric("متوسط أعمار العينة", f"{int(df['Age'].mean())} سنة")
        with col4:
            st.metric("متوسط الكوليسترول", f"{int(df['Cholesterol'].mean())} mg/dL")

        st.markdown("---")

        # عرض جدول البيانات تفاعلياً
        st.subheader("📋 أسطر البيانات الأولى (Preview)")
        num_rows = st.slider("اختر عدد الأسطر المراد عرضها:", 5, 50, 10)
        st.dataframe(df.head(num_rows))

        # معلومات الأعمدة والنوع
        st.subheader("📊 الوصف الإحصائي للمؤشرات الرقمية")
        st.dataframe(df.describe())


    # الرسوم البيانية التفاعلية

    elif page == "الرسوم البيانية التفاعلية":
        st.title("📈 التحليل البصري والرسوم البيانية")
        st.write("استكشف العلاقات بين المؤشرات الطبية المختلفة وحالة الإصابة بأزمة قلبية.")

        # اختيار نوع الرسم التفاعلي
        chart_select = st.selectbox("اختر نوع التحليل الذي تريد رؤيته:",
                                    ["توزيع الأعمار حسب الإصابة",
                                     "العلاقة بين مؤشر كتلة الجسم (BMI) وضغط الدم",
                                     "تأثير نمط الحياة (التدخين / السكري / الضغط)"])

        if chart_select == "توزيع الأعمار حسب الإصابة":
            st.subheader("توزيع الأعمار للمصابين وغير المصابين")
            fig, ax = plt.subplots(figsize=(10, 4))
            sns.histplot(data=df, x='Age', hue='Outcome', multiple='stack', bins=20, palette='Set2', ax=ax)
            st.pyplot(fig)

        elif chart_select == "العلاقة بين مؤشر كتلة الجسم (BMI) وضغط الدم":
            st.subheader("مخطط التشتت لـ BMI مقابل ضغط الدم")
            fig, ax = plt.subplots(figsize=(10, 5))
            # أخذ عينة أصغر للرسم لتسريع الأداء نظراً لضخامة البيانات
            sample_df = df.sample(1000, random_state=42)
            sns.scatterplot(data=sample_df, x='BMI', y='BloodPressure', hue='Outcome', alpha=0.7, ax=ax)
            st.pyplot(fig)

        elif chart_select == "تأثير نمط الحياة (التدخين / السكري / الضغط)":
            st.subheader("مقارنة نسب الإصابة بناءً على الاختيار")
            feature = st.radio("اختر المؤشر الحياتي:", ["Smoker", "Diabetes", "Hypertension"], horizontal=True)
            fig, ax = plt.subplots(figsize=(8, 4))
            sns.countplot(data=df, x=feature, hue='Outcome', palette='pastel', ax=ax)
            st.pyplot(fig)

    # ----------------------------------------------------
    # الصفحة الثالثة: نموذج التنبؤ الذكي
    # ----------------------------------------------------
    elif page == "نموذج التنبؤ الذكي":
        st.title(" التنبؤ الذكي بفرص الإصابة بأزمة قلبية")
        st.write("أدخل البيانات الحيوية للمريض أدناه ليقوم النموذج بحساب احتمالية الإصابة.")

        # إنشاء مدخلات تفاعلية للمستخدم (Form Inputs) بناءً على أعمدة ملفك
        st.subheader("الرجاء إدخال المؤشرات الحيوية للمريض:")

        col_in1, col_in2, col_in3 = st.columns(3)
        with col_in1:
            age = st.number_input("العمر (Age):", min_value=1, max_value=120, value=45)
            gender = st.selectbox("الجنس (Gender):", ["Male", "Female"])
            chol = st.slider("مستوى الكوليسترول (Cholesterol):", 100, 400, 200)
        with col_in2:
            bp = st.slider("ضغط الدم (Blood Pressure):", 80, 200, 120)
            hr = st.slider("معدل ضربات القلب (Heart Rate):", 50, 150, 75)
            bmi = st.number_input("مؤشر كتلة الجسم (BMI):", min_value=10.0, max_value=50.0, value=25.0)
        with col_in3:
            smoker = st.selectbox("هل المريض مدخن؟ (Smoker):", ["نعم (1)", "لا (0)"])
            diabetes = st.selectbox("هل يعاني من السكري؟ (Diabetes):", ["نعم (1)", "لا (0)"])
            diet = st.selectbox("النظام الغذائي (Diet):", ["Healthy", "Moderate", "Unhealthy"])

        st.markdown("---")

        # زر تشغيل التنبؤ
        if st.button("🚀 ابدأ تحليل الحالة والتنبؤ"):
            with st.spinner("جاري تدريب النموذج وحساب النتيجة الطبية..."):

                # تجهيز سريع للبيانات لتدريب نموذج بسيط (نفس أسلوب الدكتورة لربط المدخلات بالـ Predict)
                # قمنا باختيار أهم 5 أعمدة رقمية لتسريع التدريب الفوري أمام المستخدم
                X = df[['Age', 'Cholesterol', 'BloodPressure', 'HeartRate', 'BMI']]
                y = df['Outcome'].apply(lambda x: 1 if x == 'Heart Attack' else 0)

                # تدريب خوارزمية التعلم الآلي
                model = RandomForestClassifier(n_estimators=50, random_state=42)
                model.fit(X, y)

                # تجهيز مدخلات المستخدم الحالية للتنبؤ بها
                user_data = np.array([[age, chol, bp, hr, bmi]])
                prediction = model.predict(user_data)
                prediction_proba = model.predict_proba(user_data)[0][1]

                # عرض النتيجة بشكل مرئي جذاب للمستخدم
                st.subheader(" النتيجة المتوقعة:")
                if prediction[0] == 1:
                    st.error(f"تحذير: المريض يمتلك مؤشرات حرجة قد تؤدي إلى **أزمة قلبية** (نسبة الاحتمالية: {prediction_proba*100:.1f}%)")
                else:
                    st.success(f" مبروك: المؤشرات طبيعية والمريض في حالة **آمنة ومستقرة** (نسبة الاحتمالية: {(1-prediction_proba)*100:.1f}%)")

except FileNotFoundError:
    st.error(" خطأ: لم يتم العثور على ملف `data88.csv`. تأكد من رفعه في قائمة الملفات الجانبية في Colab.")

Overwriting app.py


In [16]:
!pip install streamlit -q
!npm install -g localtunnel -q

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 1s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹

In [18]:
!wget -qO- ipv4.icanhazip.com

print("\n")
!streamlit run app.py & npx localtunnel --port 8501

34.145.136.95


⠙⠹

⠸⠼⠴your url is: https://ninety-seals-refuse.loca.lt
2026-06-21 17:05:26.645 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.145.136.95:8501

  Stopping...
^C
